<a href="https://colab.research.google.com/github/denisejroth/pixels-frames-transformers/blob/main/part1_pixels/03_handcrafted_features/notebooks/03_handcrafted_features_solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pixels, Frames & Transformers
## Module 03: Handcrafted Features · Solutions

**A Self-Paced Course in Computational Image and Video Analysis**  
Denise J. Roth · Strategic Communication Group · Wageningen University & Research

---

This is the **solutions** version. Every ✏️ exercise is filled in with one possible answer and a short comment. There is usually more than one good way to solve an exercise, and your numbers may differ a little from run to run, so read the comments as "what to look for" rather than exact results.

Before deep learning, computer vision meant designing measurements by hand. Many of them are
still the right tool in social science: they are transparent, cheap, and often closer to a
theoretical construct than the output of a deep model. How bright is a photo? How colorful is a
cereal box? How many faces are in a protest picture, and how much text is on a package? In this
notebook you compute such measurements and ask, each time, what they actually capture.

In this notebook you will:

- measure light (brightness, contrast) and color (saturation, colorfulness, dominant colors)
- measure visual complexity and symmetry
- count faces with a classic face detector, and check it against images where you know the answer
- read visible text with OCR and measure how much text an image carries
- combine everything into one feature table, which module 04 uses to train a classifier
- test whether a feature agrees with your own judgment, and think about what it measures

**Data.** The three collections from modules 01 and 02: breakfast cereal packaging from Open Food
Facts, photos of demonstrations and protests in the Netherlands from Wikimedia Commons, and
official portraits of members of the US Congress. A helper file from the course repository
downloads them for you, so you don't need to have run module 02.

This notebook runs on a normal CPU runtime. Downloading the images takes about ten minutes the
first time.

## 0. Setup

We need **Tesseract**, an open source OCR engine, with its English and Dutch language data, and
`pytesseract` to call it from Python. Everything else is preinstalled in Colab: OpenCV (`cv2`)
for edges and faces, scikit-learn for clustering colors, and SciPy for correlations.

In [ ]:
!apt-get -qq install -y tesseract-ocr tesseract-ocr-nld > /dev/null
!pip install -q pytesseract

In [ ]:
import io
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
import cv2
import pytesseract
from PIL import Image, ImageOps
from scipy.stats import spearmanr
from sklearn.cluster import KMeans

SEED = 42
HEADERS = {"User-Agent": "PixelsFramesTransformers/1.0 (https://github.com/denisejroth/pixels-frames-transformers) course-notebook"}

def load_rgb(path, max_side=500):
    """Open an image upright, in RGB, no larger than max_side pixels on its longest side."""
    img = ImageOps.exif_transpose(Image.open(path)).convert("RGB")
    img.thumbnail((max_side, max_side))
    return img

def show_grid(paths, labels=None, n_cols=8, size=1.6):
    """Show images in a grid, with an optional label under each."""
    paths = list(paths)
    labels = list(labels) if labels is not None else [""] * len(paths)
    n_rows = max(1, int(np.ceil(len(paths) / n_cols)))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * size, n_rows * size * 1.2), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for ax, path, label in zip(axes.flat, paths, labels):
        ax.imshow(load_rgb(path, 300))
        ax.set_title(str(label)[:24], fontsize=7)
    plt.tight_layout()
    plt.show()

print("Setup complete!")

The course's data helper (`course_data.py`) contains the collection code from modules 01 and 02.
We download it once and use it to fetch the three collections. Each function returns a
**manifest**: one row per image with its local path, source, license and author.

In [ ]:
COURSE_DATA_URL = "https://raw.githubusercontent.com/denisejroth/pixels-frames-transformers/main/shared/course_data.py"
if not Path("course_data.py").exists():
    response = requests.get(COURSE_DATA_URL, headers=HEADERS, timeout=30)
    response.raise_for_status()  # stop if the download failed, rather than saving an error page
    Path("course_data.py").write_text(response.text)
import course_data

packaging = course_data.get_packaging_corpus()
protests = course_data.get_protest_corpus()
portraits = course_data.get_congress_portraits(n=200)

In [ ]:
show_grid(packaging["local_path"].head(16), packaging["product_name"].fillna("").head(16))

## 1. Light: brightness and contrast

The simplest measures work on the grayscale image from module 01.

- **Brightness** is the average gray value, from 0 (black) to 255 (white).
- **Contrast** is the standard deviation of the gray values: low when everything has a similar
  tone, high when dark and light areas sit side by side.
- The **share of clipped pixels** (almost black or almost white) shows where a photo has lost
  detail, for example because of a flash or harsh sunlight.

In [ ]:
def light_features(img):
    gray = np.asarray(img.convert("L"), dtype=float)
    return {
        "brightness": gray.mean(),
        "contrast": gray.std(),
        "share_dark": (gray < 30).mean(),
        "share_bright": (gray > 225).mean(),
    }

light = pd.DataFrame([light_features(load_rgb(p)) for p in packaging["local_path"]])
packaging = pd.concat([packaging.drop(columns=light.columns, errors="ignore"), light], axis=1)
packaging[["product_name", "brightness", "contrast", "share_dark", "share_bright"]].describe().round(2)

In [ ]:
order = packaging.sort_values("brightness")
print("Darkest packages")
show_grid(order["local_path"].head(8), order["brightness"].round(0).head(8))
print("Brightest packages")
show_grid(order["local_path"].tail(8), order["brightness"].round(0).tail(8))

Look at what drives the brightest scores: often a white or light background around the product
rather than the package itself. Volunteers photograph products in very different settings, and anything in
the frame that is not the object of interest becomes part of the measurement. This is the same
lesson as in module 01: a measure of "the image" is partly a measure of how it was produced.

> **✏️ Exercise 1**
>
> Compare brightness and contrast between products with a Nutri-Score of **a or b** and products
> with **d or e** (leave out the rest and products without a grade).
>
> 1. Make a box plot of each feature for the two groups.
> 2. Report the group medians. Is there a difference, and would you be comfortable interpreting it
>    as a marketing strategy? What else could explain it?

In [ ]:
# ✅ Solution
graded = packaging[packaging["nutriscore_grade"].isin(list("abde"))].copy()
graded["nutri_group"] = np.where(graded["nutriscore_grade"].isin(["a", "b"]), "a/b", "d/e")

fig, axes = plt.subplots(1, 2, figsize=(8, 3.5))
for ax, feature in zip(axes, ["brightness", "contrast"]):
    graded.boxplot(column=feature, by="nutri_group", ax=ax)
    ax.set_title(feature)
    ax.set_xlabel("Nutri-Score")
plt.suptitle("")
plt.tight_layout()
plt.show()

print(graded.groupby("nutri_group")[["brightness", "contrast"]].median().round(1))
print(graded["nutri_group"].value_counts())

# Differences between the groups tend to be modest and the spread within each group is large.
# Even a clear difference would be hard to interpret: product types differ (plain oats come in
# simple bags, children's cereals in printed boxes), backgrounds and lighting differ per
# contributor, and the groups are small. Treat a difference here as a lead to investigate, not
# as evidence of a strategy.

## 2. Color: saturation, colorfulness and dominant colors

**Saturation** is the average of the S channel in HSV (module 01): how far colors are from gray.

**Colorfulness** is a measure proposed by Hasler and Süsstrunk (2003), who designed it to match
how colorful people judge a photo to be. It uses two opponent color axes, red against green
(`rg = R - G`) and yellow against blue (`yb = (R + G) / 2 - B`), and combines how much they vary
across the image with how far they are from neutral on average:

`colorfulness = sqrt(sd(rg)² + sd(yb)²) + 0.3 × sqrt(mean(rg)² + mean(yb)²)`

A gray image scores 0. A photo with many different strong colors scores high; a photo that is
uniformly tinted scores lower than its saturation would suggest. In their study, a score of about
15 matched what people called "slightly colorful", about 45 "averagely colorful" and about 110
"extremely colorful".

In [ ]:
def colorfulness(img):
    """Hasler & Süsstrunk (2003) colorfulness of an RGB image."""
    arr = np.asarray(img, dtype=float)
    r, g, b = arr[:, :, 0], arr[:, :, 1], arr[:, :, 2]
    rg = r - g
    yb = 0.5 * (r + g) - b
    spread = np.sqrt(rg.std() ** 2 + yb.std() ** 2)
    shift = np.sqrt(rg.mean() ** 2 + yb.mean() ** 2)
    return spread + 0.3 * shift

def color_features(img):
    hsv = np.asarray(img.convert("HSV"), dtype=float)
    return {"saturation": hsv[:, :, 1].mean(), "colorfulness": colorfulness(img)}

# A quick check: a gray image should score 0, a rainbow should score high
gray_test = Image.new("RGB", (100, 100), (128, 128, 128))
rainbow = Image.fromarray(np.uint8(plt.cm.hsv(np.linspace(0, 1, 100))[None, :, :3].repeat(100, axis=0) * 255))
print(f"gray: {colorfulness(gray_test):.1f}   rainbow: {colorfulness(rainbow):.1f}")

In [ ]:
color = pd.DataFrame([color_features(load_rgb(p)) for p in packaging["local_path"]])
packaging = pd.concat([packaging.drop(columns=color.columns, errors="ignore"), color], axis=1)

order = packaging.sort_values("colorfulness")
sample_idx = np.linspace(0, len(order) - 1, 8).astype(int)
print("From least to most colorful (evenly spaced through the ranking)")
show_grid(order["local_path"].iloc[sample_idx], order["colorfulness"].round(0).iloc[sample_idx])

> **✏️ Exercise 2**
>
> 1. Compute saturation and colorfulness for the **protest** photos too (add them as columns to
>    `protests`), and compare the distributions of colorfulness for packaging and protests in one
>    histogram.
> 2. Show the six most colorful protest photos. What makes them colorful: the protest itself
>    (flags, banners, clothing) or something else?

In [ ]:
# ✅ Solution
protest_color = pd.DataFrame([color_features(load_rgb(p)) for p in protests["local_path"]])
protests = pd.concat([protests.drop(columns=protest_color.columns, errors="ignore"), protest_color], axis=1)

plt.figure(figsize=(6, 3.5))
plt.hist(packaging["colorfulness"], bins=30, alpha=0.6, label="packaging")
plt.hist(protests["colorfulness"], bins=30, alpha=0.6, label="protests")
plt.xlabel("colorfulness")
plt.ylabel("number of images")
plt.legend()
plt.show()

top = protests.sort_values("colorfulness", ascending=False).head(6)
show_grid(top["local_path"], top["colorfulness"].round(0), n_cols=6, size=2.2)

# Packaging is designed to stand out on a shelf and is usually more colorful than documentary
# photos. Among the protest photos, the most colorful ones often owe it to flags, banners, smoke
# or colored clothing, but sometimes to things unrelated to the protest: a blue sky, a sunset,
# or a strong color cast from street lighting. A single number can't tell those apart.

Colorfulness says *how much* color there is, not *which* colors. To describe the palette, we
group the pixels by color with **k-means clustering**: each pixel is a point in RGB space, and
k-means finds `k` cluster centers that summarize the colors, plus the share of pixels in each.
We cluster a downscaled copy, which is much faster and hardly changes the result.

In [ ]:
def dominant_colors(img, k=5):
    """Return the k dominant colors (as 0-255 RGB rows) and the share of pixels for each."""
    small = np.asarray(img.resize((80, 80)), dtype=float).reshape(-1, 3)
    km = KMeans(n_clusters=k, n_init=4, random_state=SEED).fit(small)
    shares = np.bincount(km.labels_, minlength=k) / len(km.labels_)
    order = np.argsort(shares)[::-1]
    return km.cluster_centers_[order], shares[order]

def show_palette(path, k=5):
    img = load_rgb(path)
    colors, shares = dominant_colors(img, k)
    fig, axes = plt.subplots(1, 2, figsize=(6, 2.5), gridspec_kw={"width_ratios": [1, 2]})
    axes[0].imshow(img)
    axes[0].axis("off")
    left = 0
    for c, s in zip(colors, shares):
        axes[1].barh(0, s, left=left, color=c / 255, edgecolor="white")
        left += s
    axes[1].set_xlim(0, 1)
    axes[1].set_yticks([])
    axes[1].set_xlabel("share of pixels")
    plt.tight_layout()
    plt.show()

show_palette(packaging.sort_values("colorfulness")["local_path"].iloc[-1])

> **✏️ Exercise 3**
>
> Show the palette of the **least** colorful package and of the **most** colorful one, first with
> `k=3` and then with `k=8`. How large is the share of the single most common color in each? How
> does the choice of `k` change the story the palette tells?

In [ ]:
# ✅ Solution
least = packaging.sort_values("colorfulness")["local_path"].iloc[0]
most = packaging.sort_values("colorfulness")["local_path"].iloc[-1]
for k in [3, 8]:
    for name, path in [("least colorful", least), ("most colorful", most)]:
        colors, shares = dominant_colors(load_rgb(path), k)
        print(f"k={k}, {name}: top color covers {shares[0]:.0%} of the image")
        show_palette(path, k)

# The least colorful package is usually dominated by one or two neutral tones (often including
# the background), so its top color covers a large share whatever k is. The most colorful one
# spreads over several strong colors. With k=3, k-means must merge distinct colors into one
# average, which can produce a muddy color that appears nowhere in the image; with k=8, small
# accents (a logo, a mascot's eyes) get their own cluster. Like the number of topics in a topic
# model, k is a choice you should report and check for robustness.

## 3. Composition and complexity

Three more measures that have been used in research on visual attention and aesthetics:

- **Edge density**: the share of pixels on an edge, found with the Canny edge detector. Busy
  images with lots of detail and text have many edges.
- **Compression complexity**: how many bytes per pixel a JPEG needs to store the image. Simple
  images compress well; cluttered ones don't. It is a crude but surprisingly useful measure of
  visual clutter.
- **Symmetry**: how similar the image is to its own mirror image, from 0 to 1.

In [ ]:
def composition_features(img):
    gray = np.asarray(img.convert("L"))
    edges = cv2.Canny(gray, 100, 200)
    buffer = io.BytesIO()
    img.save(buffer, format="JPEG", quality=75)
    mirrored = gray[:, ::-1].astype(float)
    return {
        "edge_density": (edges > 0).mean(),
        "bytes_per_pixel": buffer.tell() / (img.width * img.height),
        "symmetry": 1 - np.abs(gray.astype(float) - mirrored).mean() / 255,
    }

example = load_rgb(packaging["local_path"].iloc[0])
fig, axes = plt.subplots(1, 2, figsize=(6, 3.5))
axes[0].imshow(example)
axes[1].imshow(cv2.Canny(np.asarray(example.convert("L")), 100, 200), cmap="gray")
axes[1].set_title("Canny edges", fontsize=9)
for ax in axes:
    ax.axis("off")
plt.show()
print(composition_features(example))

In [ ]:
comp = pd.DataFrame([composition_features(load_rgb(p)) for p in packaging["local_path"]])
packaging = pd.concat([packaging.drop(columns=comp.columns, errors="ignore"), comp], axis=1)

feature_cols = ["brightness", "contrast", "saturation", "colorfulness", "edge_density", "bytes_per_pixel", "symmetry"]
corr = packaging[feature_cols].corr(method="spearman")

plt.figure(figsize=(5.5, 4.5))
plt.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
plt.colorbar(label="Spearman correlation")
plt.xticks(range(len(feature_cols)), feature_cols, rotation=45, ha="right")
plt.yticks(range(len(feature_cols)), feature_cols)
for i in range(len(feature_cols)):
    for j in range(len(feature_cols)):
        plt.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=7)
plt.title("How the features relate (packaging)")
plt.tight_layout()
plt.show()

Some features are strongly correlated: edge density and bytes per pixel both respond to detail,
and saturation and colorfulness both respond to strong colors. Correlated features are not
wrong, but they are not independent pieces of evidence either. If two of them "show" the same
effect, you have one finding, not two.

If you need more measures of this kind (composition, figure-ground contrast, visual balance and
dozens more), the [`athec`](https://github.com/yilangpeng/athec) library (Peng, 2022) collects
them for communication research, with references for each.

## 4. Faces with a classic detector

Whether an image shows people, how many, and how large, matters for many questions: personalized
versus impersonal political images, crowds at protests, people on packaging. The **Haar cascade**
detector (Viola and Jones, 2001) was the standard face detector for over a decade. It slides
windows of different sizes across the image and checks each for simple patterns of light and
dark that faces share (eyes darker than cheeks, the bridge of the nose lighter than the eyes). It
is fast and transparent, and it has well-known blind spots, which makes it a good place to learn
how to check a detector.

OpenCV ships the trained cascade; if your installation doesn't include it, we download it from
OpenCV's repository.

In [ ]:
cascade_name = "haarcascade_frontalface_default.xml"
cascade_path = Path(cv2.data.haarcascades) / cascade_name
if not cascade_path.exists():
    cascade_path = Path(cascade_name)
    url = f"https://raw.githubusercontent.com/opencv/opencv/4.x/data/haarcascades/{cascade_name}"
    cascade_path.write_bytes(course_data.download(url).content)
face_detector = cv2.CascadeClassifier(str(cascade_path))

def detect_faces(img, scale_factor=1.1, min_neighbors=5, min_size_share=0.03):
    """Return face boxes (x, y, w, h) found in an RGB image."""
    gray = np.asarray(img.convert("L"))
    min_side = max(12, int(min(gray.shape) * min_size_share))
    faces = face_detector.detectMultiScale(gray, scaleFactor=scale_factor,
                                           minNeighbors=min_neighbors, minSize=(min_side, min_side))
    return [tuple(int(v) for v in f) for f in faces]

def show_faces(paths, n_cols=6, **kwargs):
    paths = list(paths)
    fig, axes = plt.subplots(1, len(paths), figsize=(2.2 * len(paths), 2.6), squeeze=False)
    for ax, path in zip(axes.flat, paths):
        img = load_rgb(path)
        ax.imshow(img)
        boxes = detect_faces(img, **kwargs)
        for (x, y, w, h) in boxes:
            ax.add_patch(plt.Rectangle((x, y), w, h, fill=False, color="yellow", linewidth=2))
        ax.set_title(f"{len(boxes)} face(s)", fontsize=8)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

Before using a detector on data where you don't know the answer, test it where you do. Every
official portrait shows exactly **one** face, so the portraits are a free validation set.

In [ ]:
portraits["n_faces"] = [len(detect_faces(load_rgb(p))) for p in portraits["local_path"]]
print(portraits["n_faces"].value_counts().sort_index())
print(f"Exactly one face found in {(portraits['n_faces'] == 1).mean():.0%} of portraits")

misses = portraits[portraits["n_faces"] != 1]
if len(misses):
    show_faces(misses["local_path"].head(6))

The detector finds exactly one face in the large majority of portraits, but not in all of them.
Look at the failures: extra "faces" usually sit on a tie, a collar or the stripes of a flag, and
a face turned to the side can be missed altogether. Even on the easiest possible images, a few
percent of the counts are wrong.

> **✏️ Exercise 4**
>
> The detector has two main settings. `min_neighbors` is how many overlapping detections a window
> needs before it counts as a face (higher means stricter), and `scale_factor` is how much the
> window grows between passes (smaller means more sizes are tried: slower, but more thorough).
>
> 1. Run the detector on the portraits with `min_neighbors` set to 2, 5 and 10 and `scale_factor`
>    set to 1.05 and 1.3 (six combinations). For each, report the share of portraits with 0 faces,
>    exactly 1 face, and more than 1 face.
> 2. Which setting would you choose, and what does the trade-off look like?

In [ ]:
# ✅ Solution
results = []
for min_neighbors in [2, 5, 10]:
    for scale_factor in [1.05, 1.3]:
        counts = np.array([len(detect_faces(load_rgb(p), scale_factor=scale_factor, min_neighbors=min_neighbors))
                           for p in portraits["local_path"]])
        results.append({"min_neighbors": min_neighbors, "scale_factor": scale_factor,
                         "no face": (counts == 0).mean(), "one face": (counts == 1).mean(),
                         "more than one": (counts > 1).mean()})
print(pd.DataFrame(results).round(2))

# Low min_neighbors finds every face but adds false "faces" in ties, flags and backgrounds (more
# than one per portrait). High min_neighbors removes false alarms but starts to miss real faces,
# especially with a coarse scale_factor. The middle settings usually find exactly one face in
# nearly every portrait. Note that this is an easy test: frontal, well-lit, centered faces. A
# setting tuned here will do worse on crowds, as the next section shows.

Now the hard case: protest photos, with crowds, faces in profile, small faces in the distance,
hats, masks and banners. We record the number of faces and the share of the image covered by the
largest face, a common measure of how "personal" or close-up an image is.

In [ ]:
def face_features(img):
    boxes = detect_faces(img)
    areas = [w * h for (_, _, w, h) in boxes]
    return {"n_faces": len(boxes), "largest_face_share": max(areas) / (img.width * img.height) if areas else 0.0}

face = pd.DataFrame([face_features(load_rgb(p)) for p in protests["local_path"]])
protests = pd.concat([protests.drop(columns=face.columns, errors="ignore"), face], axis=1)
print(protests["n_faces"].describe().round(2))

show_faces(protests.sort_values("n_faces", ascending=False)["local_path"].head(6))

Look closely at the crowd photos: the detector finds some faces, misses many (anyone turned away
or far off), and occasionally marks a pattern on a banner as a face. **Face counts from a classic
detector are a lower bound on the number of people, with an unknown amount of noise**, and the
error depends on the type of scene. That is fine for some questions (close-up versus distant
images) and not good enough for others (crowd size). Module 12 returns to counting people with
modern detectors.

## 5. Visible text with OCR

Packaging, posters, memes and protest signs carry text, and how much text an image contains is
itself a feature. Tesseract finds words and returns, for each, its position and a confidence
score from 0 to 100. We keep words with a confidence above 60 and at least two characters, which
removes most of the noise that OCR produces on pictures.

In [ ]:
def ocr_words(img, lang="eng+nld", min_conf=60):
    """Return a DataFrame of words found by Tesseract, with position and confidence."""
    data = pytesseract.image_to_data(img, lang=lang, output_type=pytesseract.Output.DATAFRAME)
    data = data.dropna(subset=["text"])
    data["text"] = data["text"].astype(str).str.strip()
    return data[(data["conf"] > min_conf) & (data["text"].str.len() >= 2)]

def text_features(img, **kwargs):
    words = ocr_words(img, **kwargs)
    return {"n_words": len(words),
            "text_area_share": (words["width"] * words["height"]).sum() / (img.width * img.height),
            "words": " ".join(words["text"])}

example = load_rgb(packaging["local_path"].iloc[0], max_side=800)
words = ocr_words(example)
fig, ax = plt.subplots(figsize=(4, 5))
ax.imshow(example)
for row in words.itertuples():
    ax.add_patch(plt.Rectangle((row.left, row.top), row.width, row.height, fill=False, color="red", linewidth=1.5))
ax.axis("off")
plt.show()
print("Words found:", " | ".join(words["text"]))

In [ ]:
text = pd.DataFrame([text_features(load_rgb(p, max_side=800)) for p in packaging["local_path"]])
packaging = pd.concat([packaging.drop(columns=text.columns, errors="ignore"), text], axis=1)
print(packaging["n_words"].describe().round(1))
packaging[["product_name", "n_words", "words"]].head(10)

> **✏️ Exercise 5**
>
> OCR results depend on settings you might not think about.
>
> 1. For 20 packages, compare the number of words found with `lang="eng"` and with `lang="eng+nld"`.
>    Print a few words that only the Dutch-aware version finds.
> 2. Tesseract's documentation warns that accuracy drops when lowercase letters are less than
>    about 20 pixels tall, and that below 10 pixels little is recognized. For the same 20
>    packages, compare `max_side=400` with `max_side=1200` (using `lang="eng+nld"`). Does a larger
>    image help?

In [ ]:
# ✅ Solution
subset = packaging["local_path"].head(20)
rows = []
for path in subset:
    small, large = load_rgb(path, max_side=400), load_rgb(path, max_side=1200)
    medium = load_rgb(path, max_side=800)
    eng = set(ocr_words(medium, lang="eng")["text"])
    both = set(ocr_words(medium, lang="eng+nld")["text"])
    rows.append({"eng": len(eng), "eng+nld": len(both), "only_with_nld": sorted(both - eng)[:4],
                 "max_side 400": len(ocr_words(small)), "max_side 1200": len(ocr_words(large))})
comparison = pd.DataFrame(rows)
print(comparison[["eng", "eng+nld", "max_side 400", "max_side 1200"]].sum())
comparison.head(10)

# Adding Dutch usually helps on Dutch packaging: words like "volkoren" or "vezels" are recognized
# with higher confidence, so more of them pass the confidence filter. Resolution matters even
# more: at 400 pixels, small print on a package is only a few pixels tall and Tesseract finds
# little; enlarging the image often finds more words, but only if the original photo had that
# detail. Both are choices to report, because they change the measured "amount of text".

## 6. One feature table

Finally we wrap everything in one function and build a feature table for the protest photos, the
input for module 04. Writing it as a single function has a practical benefit: the same
measurement is applied in the same way to every image, and you can report exactly what it does.

In [ ]:
def extract_features(path):
    img = load_rgb(path)
    features = {**light_features(img), **color_features(img), **composition_features(img), **face_features(img)}
    text_info = text_features(load_rgb(path, max_side=800))
    features.update(n_words=text_info["n_words"], text_area_share=text_info["text_area_share"])
    return features

protest_features = pd.DataFrame([extract_features(p) for p in protests["local_path"]])
protest_features.insert(0, "title", protests["title"].values)
protest_features.insert(1, "local_path", protests["local_path"].values)
protest_features.to_csv("protest_features.csv", index=False)
print(protest_features.shape)
protest_features.describe().T.round(2)

## 7. Do the features measure what we think they measure?

Every feature in this notebook is precisely defined, which makes it easy to forget that
precision is not validity. A feature is only useful if it captures the concept you care about.
Two kinds of evidence help:

- **Checks against known truth**, like the portraits for the face detector.
- **Checks against human judgment**: do people's ratings agree with the feature? Hasler and
  Süsstrunk validated colorfulness this way on their photos, but your images (packaging, protests,
  memes) may behave differently.

Let's run a small version of the second check yourself.

In [ ]:
to_rate = packaging.sample(12, random_state=1).reset_index(drop=True)
show_grid(to_rate["local_path"], [f"#{i + 1}" for i in range(12)], n_cols=6, size=2.2)

> **✏️ Exercise 6**
>
> Rate the 12 packages above for **how colorful they look to you**, from 1 (not colorful at all) to
> 5 (extremely colorful). Don't look at the scores first. Put your ratings in a list `my_ratings`,
> in the order #1 to #12.
>
> Then compute the Spearman correlation between your ratings and `colorfulness`, `saturation` and
> `brightness`. Which feature agrees best with your judgment? Where do you and the measure disagree
> most, and why?

In [ ]:
# ✅ Solution
# Example ratings given when this notebook was written. Rate the images you see yourself: they
# may differ from these if the collection has changed.
my_ratings = [4, 2, 5, 3, 1, 4, 3, 5, 2, 3, 4, 1]

to_rate["my_rating"] = my_ratings
for feature in ["colorfulness", "saturation", "brightness"]:
    rho, p = spearmanr(to_rate["my_rating"], to_rate[feature])
    print(f"{feature:<13} rho = {rho:5.2f}  (p = {p:.2f})")

to_rate["gap"] = to_rate["my_rating"].rank() - to_rate["colorfulness"].rank()
worst = to_rate.reindex(to_rate["gap"].abs().sort_values(ascending=False).index).head(3)
show_grid(worst["local_path"], [f"me: {r}, score: {c:.0f}" for r, c in zip(worst["my_rating"], worst["colorfulness"])],
          n_cols=3, size=2.4)

# With 12 images the estimate is rough, but colorfulness typically agrees with people better than
# saturation, and brightness hardly at all. Disagreements often come from backgrounds (a colorful
# package on a gray table), from color casts in the photo, or from a single large block of one
# strong color, which people find colorful while the formula, which rewards variety, does not.

> **✏️ Exercise 7**
>
> Suppose you want to use **colorfulness** in a study of how cereal brands target children.
>
> 1. What construct would colorfulness stand in for in that study? And in a study of protest
>    photos, what might the same number stand for?
> 2. List three things other than the package design that could change a package's colorfulness
>    score in this collection.
> 3. How would you validate the measure before using it in your study?

**✅ One possible answer**

1. For cereals, colorfulness might stand in for **child-oriented design**: bright, varied colors
   meant to attract children's attention. In protest photos the same number might reflect
   **visual intensity** or the presence of symbols (flags, banners, smoke), or simply the weather.
   One measure, different constructs, depending on the theory you bring.
2. The **background and setting** of the photo (a colorful table cloth, a supermarket shelf),
   **lighting and white balance** (a yellow kitchen lamp tints everything), **camera and
   compression** (phone processing boosts saturation), and **cropping** (how much of the frame is
   package versus background). Also the package's **side**: a front photo versus a photo of the
   back with the nutrition table.
3. Code a sample of packages for the construct itself ("Is this package designed to appeal to
   children?", with a codebook and two coders) and check how well colorfulness predicts those
   codes, ideally next to other features (characters, cartoon fonts). Standardize the images first
   (crop to the package) or at least check that results hold for the subset with plain
   backgrounds. Report the validation, not just the feature.

## Wrap-up

You have seen that:

- brightness, contrast, saturation and colorfulness are easy to compute and to report, and they
  respond to everything in the frame, including backgrounds and lighting;
- dominant colors via k-means describe a palette, with `k` as a choice to report;
- edge density and compression size measure visual complexity, and correlated features are not
  independent evidence;
- a classic face detector can be validated on images with a known answer, and works much worse on
  crowds than on portraits;
- OCR finds visible text, with results that depend on the language setting and the resolution;
- a feature is only as good as its validation against the construct you care about.

**Next:** in module 04 you code images by hand with a codebook, measure agreement, and train a
first classifier on the features from this notebook.